## Getting Dataset

In [1]:
import os
import shutil

animal_root_dir = 'Animals'

# List of known animal categories (based on unzipped folder names)
animal_categories = [
    'Clams', 'Corals', 'Crabs', 'Dolphin', 'Eel', 'Fish', 'Jelly Fish', 'Lobster',
    'Nudibranchs', 'Octopus', 'Otter', 'Penguin', 'Puffers', 'Sea Rays', 'Sea Urchins',
    'Seahorse', 'Seal', 'Sharks', 'Shrimp', 'Squid', 'Starfish', 'Turtle_Tortoise', 'Whale'
]

print('Performing proactive cleanup before data setup...')

# Remove the target 'Animals' directory if it exists
if os.path.exists(animal_root_dir) and os.path.isdir(animal_root_dir):
    print(f"  Removing existing organized '{animal_root_dir}' directory...")
    shutil.rmtree(animal_root_dir)
    print(f"  '{animal_root_dir}' removed.")

# Remove any raw unzipped animal folders from the root directory
print('  Cleaning up any remaining raw unzipped animal folders from root directory...')
for category in animal_categories:
    category_path = os.path.join('.', category)
    if os.path.exists(category_path) and os.path.isdir(category_path):
        shutil.rmtree(category_path)
        print(f"    Removed raw unzipped folder: '{category_path}'")
print('  Cleanup of raw folders complete.')

print('Proactive cleanup finished.')

Performing proactive cleanup before data setup...
  Cleaning up any remaining raw unzipped animal folders from root directory...
  Cleanup of raw folders complete.
Proactive cleanup finished.


In [2]:
!unzip Animals.zip

Streaming output truncated to the last 5000 lines.
  inflating: Seahorse/9891995103_849905035f_o.jpg  
  inflating: Seahorse/9984393_fb8ac77d6a_o.jpg  
  inflating: Seahorse/Sea Horse (1).jpg  
  inflating: Seahorse/Sea Horse (10).jpg  
  inflating: Seahorse/Sea Horse (11).jpg  
  inflating: Seahorse/Sea Horse (12).jpg  
  inflating: Seahorse/Sea Horse (13).jpg  
  inflating: Seahorse/Sea Horse (14).jpg  
  inflating: Seahorse/Sea Horse (15).jpg  
  inflating: Seahorse/Sea Horse (16).jpg  
  inflating: Seahorse/Sea Horse (17).jpg  
  inflating: Seahorse/Sea Horse (18).jpg  
  inflating: Seahorse/Sea Horse (19).jpg  
  inflating: Seahorse/Sea Horse (2).jpg  
  inflating: Seahorse/Sea Horse (20).jpg  
  inflating: Seahorse/Sea Horse (21).jpg  
  inflating: Seahorse/Sea Horse (22).jpg  
  inflating: Seahorse/Sea Horse (23).jpg  
  inflating: Seahorse/Sea Horse (24).jpg  
  inflating: Seahorse/Sea Horse (25).jpg  
  inflating: Seahorse/Sea Horse (26).jpg  
  inflating: Seahorse/Sea Horse (

In [3]:
# Create a new directory named 'Animals'
!mkdir Animals

In [4]:
# List the contents of the current directory to check for remaining unzipped folders
!ls -F

 Animals/      Eel/	      Otter/	    'Sea Rays'/      Turtle_Tortoise/
 Animals.zip   Fish/	      Penguin/	    'Sea Urchins'/   Whale/
 Clams/       'Jelly Fish'/   Puffers/	     Sharks/
 Corals/       Lobster/       sample_data/   Shrimp/
 Crabs/        Nudibranchs/   Seahorse/      Squid/
 Dolphin/      Octopus/       Seal/	     Starfish/


In [5]:
import os
import shutil

destination_folder = './Animals'

# Get all directories in the current working directory
current_dirs = [d for d in os.listdir('.') if os.path.isdir(d)]

# Define common non-animal folders to exclude from moving initially
# '.ipynb_checkpoints' might exist in Colab environment
excluded_folders = ['Animals', 'data', 'sample_data', '.ipynb_checkpoints']

print(f"Moving folders into '{destination_folder}'...")
for folder_name in current_dirs:
    if folder_name not in excluded_folders: # Exclude Animals itself and other system folders
        source_path = os.path.join('.', folder_name)
        destination_path = os.path.join(destination_folder, folder_name)
        if os.path.exists(source_path):
            print(f"  Moving '{source_path}' to '{destination_path}'")
            shutil.move(source_path, destination_path)
        else:
            print(f"  Warning: Folder '{source_path}' not found, skipping.")

# --- FIX: Remove .config folder from inside Animals directory if it exists ---
config_in_animals_path = os.path.join(destination_folder, '.config')
if os.path.exists(config_in_animals_path) and os.path.isdir(config_in_animals_path):
    print(f"Removing non-image folder: {config_in_animals_path}")
    shutil.rmtree(config_in_animals_path)
# -------------------------------------------------------------------------


print("Finished moving folders. Verifying contents of the 'Animals' directory:")
!ls -F ./Animals

Moving folders into './Animals'...
  Moving './.config' to './Animals/.config'
  Moving './Crabs' to './Animals/Crabs'
  Moving './Whale' to './Animals/Whale'
  Moving './Fish' to './Animals/Fish'
  Moving './Octopus' to './Animals/Octopus'
  Moving './Lobster' to './Animals/Lobster'
  Moving './Jelly Fish' to './Animals/Jelly Fish'
  Moving './Nudibranchs' to './Animals/Nudibranchs'
  Moving './Shrimp' to './Animals/Shrimp'
  Moving './Squid' to './Animals/Squid'
  Moving './Clams' to './Animals/Clams'
  Moving './Dolphin' to './Animals/Dolphin'
  Moving './Seahorse' to './Animals/Seahorse'
  Moving './Eel' to './Animals/Eel'
  Moving './Seal' to './Animals/Seal'
  Moving './Penguin' to './Animals/Penguin'
  Moving './Starfish' to './Animals/Starfish'
  Moving './Sea Rays' to './Animals/Sea Rays'
  Moving './Turtle_Tortoise' to './Animals/Turtle_Tortoise'
  Moving './Sharks' to './Animals/Sharks'
  Moving './Corals' to './Animals/Corals'
  Moving './Sea Urchins' to './Animals/Sea Urch

In [6]:
import torch
import torch.nn as nn

## Coding the Model

In [7]:
class DepthwiseConvBlock(nn.Module):
  def __init__(self,in_channels,out_channels,stride):
    super().__init__()

    ##depthwise convolution layer
    self.depthwise_layer = nn.Sequential(
        nn.Conv2d(in_channels,in_channels,kernel_size=3,stride=stride,padding=1,groups=in_channels,bias=False),
        nn.BatchNorm2d(in_channels),
        nn.ReLU(inplace=True)
    )

    ##pointwise layer
    self.pointwise = nn.Sequential(
        nn.Conv2d(in_channels,out_channels,kernel_size=1,stride=1,padding=0,bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True)
    )

  def forward(self,x:torch.Tensor):
    x = self.depthwise_layer(x)
    x= self.pointwise(x)
    return x


In [8]:
class Mobilenet(nn.Module):
  def __init__(self,num_classes=23):
    super().__init__()

    #stem layer
    self.conv1 = nn.Sequential(
        nn.Conv2d(3,32,kernel_size=3,stride=2,padding=1,bias=False),
        nn.BatchNorm2d(32),
        nn.ReLU(inplace=True)
    )

    #depthwise blocks
    layers = [
        DepthwiseConvBlock(32,64,1),
        DepthwiseConvBlock(64,128,2),
        DepthwiseConvBlock(128,128,1),
        DepthwiseConvBlock(128,256,2),
        DepthwiseConvBlock(256,256,1),
        DepthwiseConvBlock(256,512,2),

    ]

    ##adding five identical depthwise blocks
    for _ in range(5):
      layers.append(DepthwiseConvBlock(512,512,1))

    layers.extend([
          DepthwiseConvBlock(512,1024,2),
          DepthwiseConvBlock(1024,1024,1)
    ])
    self.depthwise_blocks= nn.Sequential(*layers)

    self.avgpool = nn.AdaptiveAvgPool2d((1,1))

    self.fc = nn.Linear(1024,num_classes)


  def forward(self,x:torch.Tensor)->torch.Tensor:
    x = self.conv1(x)
    x = self.depthwise_blocks(x)
    x = self.avgpool(x)
    x = torch.flatten(x, 1)
    x = self.fc(x)
    return x


In [9]:
model = Mobilenet()
model

Mobilenet(
  (conv1): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (depthwise_blocks): Sequential(
    (0): DepthwiseConvBlock(
      (depthwise_layer): Sequential(
        (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
      )
      (pointwise): Sequential(
        (0): Conv2d(32, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
      )
    )
    (1): DepthwiseConvBlock(
      (depthwise_layer): Sequential(
        (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=64, bias=False)
        (1)

In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [11]:
##moving model to device
model = model.to(device)
model

Mobilenet(
  (conv1): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (depthwise_blocks): Sequential(
    (0): DepthwiseConvBlock(
      (depthwise_layer): Sequential(
        (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
      )
      (pointwise): Sequential(
        (0): Conv2d(32, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
      )
    )
    (1): DepthwiseConvBlock(
      (depthwise_layer): Sequential(
        (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=64, bias=False)
        (1)

## downloading dataset

In [12]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import os

In [13]:
import torch
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms
import os
import shutil

# AlexNet expects 224x224 input images
image_size = 224

# Helper class to apply different transforms to a subset
class TransformedSubset(Subset):
    def __init__(self, dataset, indices, transform=None):
        super().__init__(dataset, indices)
        self.transform = transform

    def __getitem__(self, idx):
        # Get the original image and label from the base dataset
        # The base dataset's transform (base_dataset_transform) should yield PIL images after resize.
        original_image, label = self.dataset[self.indices[idx]]

        if self.transform is not None:
            image = self.transform(original_image)
        else:
            image = original_image # If no specific transform, use original
        return image, label

    # Required for DataLoader to work correctly when __getitem__ is overridden in Subset
    def __getitems__(self, indices):
        return [self.__getitem__(idx) for idx in indices]

# Define transforms for clarity, making them specific to the splitting strategy

# Transforms for the base dataset before splitting (only resize, keeps as PIL)
# This is applied to `full_dataset` before `random_split`.
base_dataset_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
])

# Full transform for training data (includes augmentation, then ToTensor and Normalize)
train_full_transform = transforms.Compose([
    transforms.RandomResizedCrop(image_size), # Added data augmentation
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15), # Added data augmentation
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.1), # Re-added ColorJitter for better generalization
    transforms.RandomPerspective(distortion_scale=0.2, p=0.5), # Added RandomPerspective
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

# Full transform for validation and test data (no augmentation, then ToTensor and Normalize)
val_test_full_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

# Define the dataset path
dataset_path = './Animals'

# Ensure the .config folder is removed if it exists before loading with ImageFolder
config_in_animals_path = os.path.join(dataset_path, '.config')
if os.path.exists(config_in_animals_path) and os.path.isdir(config_in_animals_path):
    print(f"Removing non-image folder: {config_in_animals_path}")
    # Use shutil.rmtree for directories
    shutil.rmtree(config_in_animals_path)

# Load the full dataset with the base_dataset_transform (to get PIL images after resize)
full_dataset = datasets.ImageFolder(
    root=dataset_path,
    transform=base_dataset_transform
)

# Get the number of classes from the dataset
num_classes_dataset = len(full_dataset.classes)
print(f"Number of classes in dataset: {num_classes_dataset}")

# Calculate split lengths
total_size = len(full_dataset)
train_size = int(0.7 * total_size)
val_size = int(0.15 * total_size)
test_size = total_size - train_size - val_size

# Perform random split on the full_dataset (which returns PIL images after resize)
seed = torch.Generator().manual_seed(42) # for reproducibility
train_subset_indices, val_subset_indices, test_subset_indices = random_split(full_dataset, [train_size, val_size, test_size], generator=seed)

# Wrap subsets with TransformedSubset to apply specific transforms
train_data_split = TransformedSubset(full_dataset, train_subset_indices.indices, transform=train_full_transform)
val_data_split = TransformedSubset(full_dataset, val_subset_indices.indices, transform=val_test_full_transform)
test_data_split = TransformedSubset(full_dataset, test_subset_indices.indices, transform=val_test_full_transform)

print(f"Train set size: {len(train_data_split)}")
print(f"Validation set size: {len(val_data_split)}")
print(f"Test set size: {len(test_data_split)}")

# Create DataLoaders for each split
BATCH_SIZE = 32 # You can adjust this batch size
NUM_WORKERS = 2 # Adjust based on your system's capabilities, changed from 4 to 2 based on UserWarning

train_loader = DataLoader(
    train_data_split,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True # Speeds up data transfer to GPU
)

val_loader = DataLoader(
    val_data_split,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_data_split,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("DataLoaders for train, validation, and test sets created successfully.")

Number of classes in dataset: 23
Train set size: 9597
Validation set size: 2056
Test set size: 2058
DataLoaders for train, validation, and test sets created successfully.


## Training

In [14]:
def train_step(model: torch.nn.Module,
                   dataloader: torch.utils.data.DataLoader,
                   loss_fn: torch.nn.Module,
                   optimizer: torch.optim.Optimizer,device: torch.device):
    # Put model in train mode
    model.train()

    # Setup train loss and train accuracy values
    train_loss, train_acc = 0, 0

    # Loop through data loader data batches
    for batch, (X, y) in enumerate(dataloader):
        # Send data to target device
        X, y = X.to(device), y.to(device)

        # 1. Forward pass
        y_pred = model(X)

        # 2. Calculate  and accumulate loss
        loss = loss_fn(y_pred, y)
        train_loss += loss.item()

        # 3. Optimizer zero grad
        optimizer.zero_grad()

        # 4. Loss backward
        loss.backward()

        # 5. Optimizer step
        optimizer.step()

        # Calculate and accumulate accuracy metrics across all batches
        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
        train_acc += (y_pred_class == y).sum().item()/len(y_pred)

    # Adjust metrics to get average loss and accuracy per batch
    train_loss = train_loss / len(dataloader)
    train_acc = train_acc / len(dataloader)
    return train_loss, train_acc

In [15]:
def test_step(model: torch.nn.Module,
                  dataloader: torch.utils.data.DataLoader,
                  loss_fn: torch.nn.Module,device: torch.device):
    # Put model in eval mode
    model.eval()

    # Setup test loss and test accuracy values
    test_loss, test_acc = 0, 0

    # Turn on inference context manager
    with torch.inference_mode():
        # Loop through DataLoader batches
        for batch, (X, y) in enumerate(dataloader):
            # Send data to target device
            X, y = X.to(device), y.to(device)

            # 1. Forward pass
            test_pred_logits = model(X)

            # 2. Calculate and accumulate loss
            loss = loss_fn(test_pred_logits, y)
            test_loss += loss.item()

            # Calculate and accumulate accuracy
            test_pred_labels = test_pred_logits.argmax(dim=1)
            test_acc += ((test_pred_labels == y).sum().item()/len(test_pred_labels))

    # Adjust metrics to get average loss and accuracy per batch
    test_loss = test_loss / len(dataloader)
    test_acc = test_acc / len(dataloader)
    return test_loss, test_acc

In [16]:
from tqdm.auto import tqdm
import copy # Import copy module for deepcopy

# 1. Take in various parameters required for training and test steps
def train(model: torch.nn.Module,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader: torch.utils.data.DataLoader,
          optimizer: torch.optim.Optimizer,
          loss_fn: torch.nn.Module = nn.CrossEntropyLoss(),
          epochs: int = 5,
          device: torch.device = 'cpu',
          patience: int = 7,
          scheduler: torch.optim.lr_scheduler._LRScheduler = None): # Added scheduler parameter

    # 2. Create empty results dictionary
    results = {"train_loss": [],
        "train_acc": [],
        "test_loss": [],
        "test_acc": []
    }

    best_val_loss = float('inf')
    epochs_no_improve = 0
    best_model_wts = copy.deepcopy(model.state_dict()) # To store the best model weights

    # 3. Loop through training and testing steps for a number of epochs
    for epoch in tqdm(range(epochs)):
        train_loss, train_acc = train_step(model=model,
                                           dataloader=train_dataloader,
                                           loss_fn=loss_fn,
                                           optimizer=optimizer,
                                           device=device) # Pass device here
        test_loss, test_acc = test_step(model=model,
            dataloader=test_dataloader,
            loss_fn=loss_fn,
            device=device) # Pass device here

        # 4. Print out what's happening
        print(
            f"Epoch: {epoch+1} | "
            f"train_loss: {train_loss:.4f} | "
            f"train_acc: {train_acc:.4f} | "
            f"test_loss: {test_loss:.4f} | "
            f"test_acc: {test_acc:.4f}"
        )

        # Step the scheduler if it's provided
        if scheduler is not None:
            # For ReduceLROnPlateau, step with the validation loss
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(test_loss)
            else:
                scheduler.step()
            print(f"Current learning rate: {optimizer.param_groups[0]['lr']}")

        # Early stopping logic
        if test_loss < best_val_loss:
            best_val_loss = test_loss
            epochs_no_improve = 0
            best_model_wts = copy.deepcopy(model.state_dict()) # Save the best model
        else:
            epochs_no_improve += 1
            print(f"Validation loss did not improve for {epochs_no_improve} epochs.")
            if epochs_no_improve == patience:
                print(f"Early stopping triggered after {epoch+1} epochs. Restoring best model weights.")
                model.load_state_dict(best_model_wts)
                break # Stop training

        # 5. Update results dictionary
        # Ensure all data is moved to CPU and converted to float for storage
        results["train_loss"].append(train_loss.item() if isinstance(train_loss, torch.Tensor) else train_loss)
        results["train_acc"].append(train_acc.item() if isinstance(train_acc, torch.Tensor) else train_acc)
        results["test_loss"].append(test_loss.item() if isinstance(test_loss, torch.Tensor) else test_loss)
        results["test_acc"].append(test_acc.item() if isinstance(test_acc, torch.Tensor) else test_acc)

    # 6. Return the filled results at the end of the epochs
    return results

In [17]:
# Set random seeds
torch.manual_seed(42)
torch.cuda.manual_seed(42)

# Set number of epochs
NUM_EPOCHS = 90

# Setup loss function and optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001) # Changed from SGD to Adam, adjusted LR

# Add a learning rate scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5)

# Start the timer
from timeit import default_timer as timer
start_time = timer()

# Train model_0
model_0_results = train(model=model,
                        train_dataloader=train_loader,
                        test_dataloader=test_loader,
                        optimizer=optimizer,
                        loss_fn=loss_fn,
                        epochs=NUM_EPOCHS,
                        device=device,
                        scheduler=scheduler) # Pass the scheduler here

# End the timer and print out how long it took
end_time = timer()
print(f"Total training time: {end_time-start_time:.3f} seconds")

  0%|          | 0/90 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 2.7967 | train_acc: 0.1721 | test_loss: 2.6781 | test_acc: 0.2044
Current learning rate: 0.001
Epoch: 2 | train_loss: 2.6088 | train_acc: 0.2091 | test_loss: 2.4705 | test_acc: 0.2565
Current learning rate: 0.001
Epoch: 3 | train_loss: 2.5000 | train_acc: 0.2358 | test_loss: 2.2957 | test_acc: 0.3036
Current learning rate: 0.001
Epoch: 4 | train_loss: 2.4489 | train_acc: 0.2590 | test_loss: 2.2871 | test_acc: 0.3061
Current learning rate: 0.001
Epoch: 5 | train_loss: 2.3751 | train_acc: 0.2784 | test_loss: 2.1537 | test_acc: 0.3377
Current learning rate: 0.001
Epoch: 6 | train_loss: 2.3226 | train_acc: 0.3004 | test_loss: 2.1012 | test_acc: 0.3536
Current learning rate: 0.001
Epoch: 7 | train_loss: 2.2790 | train_acc: 0.3015 | test_loss: 2.1043 | test_acc: 0.3631
Current learning rate: 0.001
Validation loss did not improve for 1 epochs.
Epoch: 8 | train_loss: 2.2497 | train_acc: 0.3192 | test_loss: 2.0271 | test_acc: 0.3718
Current learning rate: 0.001
Epoch: 9 |